# Hangman — seed ensemble (round 3 twin + blend)

One unattended run, roughly 3.5 hours, that does two things:

1. **Trains a twin of round 3** — identical architecture, identical 9.17M state
   buffer, only the random seed differs. Different initialisation and different
   batch order make it learn a slightly different function of the same data.
2. **Blends every available model** and picks the mix that wins on the full
   10,000-word holdout, then scores it on `test.txt` and writes a submission.

**Why a twin rather than another DAgger round.** Ensembling works by cancelling
the errors two models make independently. It works best between models of
*equal* strength. Round 2 is weaker than round 3 and was trained on a subset of
its data, so it may add nothing. A same-architecture, same-data twin is a true
equal, and the only thing separating the two is randomness — which is exactly
the error an ensemble cancels.

**Why not a round 4.** The gap between win rate on training words and on the
holdout widened 4.6 → 6.8 across rounds. More aggregation is starting to teach
the model these specific words rather than English.

| | holdout | test.txt | public LB |
|---|---|---|---|
| round 2 (8L, d256) | 65.96% | 67.776% | 67.9306 |
| round 3 (10L, d384) | 67.28% | 70.097% | 70.0334 |
| this run | ? | ? | ? |

Attach the competition, `hangman-src` and `hangman-weights`. GPU on.
**Run as Save & Run All (Commit)** — it is far too long for an interactive session.

In [ ]:
import sys, json, time, warnings
warnings.filterwarnings("ignore")

SRC = "/kaggle/input/datasets/suniljadaun/hangman-src/src"
WEIGHTS = "/kaggle/input/datasets/suniljadaun/hangman-weights"
sys.path.insert(0, SRC)

import numpy as np
import torch

from hangman.data import load_words, overlap, split_holdout, find_competition_dir
from hangman.model import HangmanNet, ModelConfig
from hangman.policies import NeuralPolicy, FallbackPolicy
from hangman.states import StateBuffer, BoardBatcher
from hangman.train import train, TrainConfig
from hangman.evaluate import evaluate, print_report
from hangman.submit import write_submission, validate_submission

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

COMP = find_competition_dir()
train_words_all = load_words(f"{COMP}/train.txt")
test_words = load_words(f"{COMP}/test.txt")
assert overlap(train_words_all, test_words)["test_words_in_train"] == 0
MAX_LEN = max(max(map(len, train_words_all)), max(map(len, test_words)))

# Same seed as every previous run: identical holdout, comparable numbers.
TRAIN_WORDS, HOLDOUT = split_holdout(train_words_all, n_holdout=10_000, seed=0)
print(len(TRAIN_WORDS), len(HOLDOUT), "MAX_LEN =", MAX_LEN)

## 1. Train the twin

Same `ModelConfig`, same buffer, `seed=7` instead of `3`. Nothing else changes.

In [ ]:
buffer = StateBuffer.load(f"{WEIGHTS}/states_round3.npz")
print(f"{len(buffer):,} states loaded")

MODEL_CFG = ModelConfig(d_model=384, n_layers=10, n_heads=8, d_ff=1536,
                        dropout=0.1, max_len=MAX_LEN)
batcher = BoardBatcher(TRAIN_WORDS, buffer, max_len=MAX_LEN,
                       batch_size=512, bucket=True, seed=7)
cfg = TrainConfig(epochs=3, batch_size=512, lr=3e-4,
                  amp=(DEVICE == "cuda"), model=MODEL_CFG, seed=7)
print(f"{len(batcher):,} batches/epoch")

t = time.time()
twin = train(batcher, cfg, device=DEVICE)
torch.save({"model": twin.state_dict(), "cfg": MODEL_CFG.__dict__},
           "/kaggle/working/hangman_r3b.pt")
print(f"trained in {(time.time()-t)/60:.0f} min")

## 2. Sanity-check the twin alone

It should land within about a point of round 3's 67.28%. If it is far below,
something went wrong in training and the blend below is not worth trusting.

In [ ]:
def load(name):
    ck = torch.load(f"{WEIGHTS}/{name}", map_location=DEVICE)
    net = HangmanNet(ModelConfig(**ck["cfg"]))
    net.load_state_dict(ck["model"])
    return net

p3  = NeuralPolicy(load("hangman_r3.pt"), DEVICE, fusion=0.6)
p2  = NeuralPolicy(load("hangman_r2.pt"), DEVICE, fusion=0.3)
p3b = NeuralPolicy(twin, DEVICE, fusion=0.6)

SOLO_R3 = 67.28   # measured, same holdout
m = evaluate(HOLDOUT, p3b, max_len=MAX_LEN)
print(f"twin alone: {m['win_rate']:.2f}%  strikes {m['mean_wrong']:.3f}"
      f"   (round 3 alone was {SOLO_R3}%)")

## 3. Find the best blend on the full holdout

Two families: round 3 + twin (the equals), and that pair further mixed with
round 2. All on 10,000 words — a 4,000-word sweep has a ~0.8-point standard
error, which is enough to pick a winner out of pure noise.

In [ ]:
candidates = {"round 3 alone": p3, "twin alone": p3b}
for w in (0.5, 0.4, 0.6):
    candidates[f"r3+twin w={w}"] = FallbackPolicy(p3, p3b, weight=w)
pair = FallbackPolicy(p3, p3b, weight=0.5)
for w in (0.8, 0.7):
    candidates[f"(r3+twin)+r2 w={w}"] = FallbackPolicy(pair, p2, weight=w)

scored = []
for name, pol in candidates.items():
    t = time.time()
    m = evaluate(HOLDOUT, pol, max_len=MAX_LEN)
    scored.append((m["win_rate"], -m["mean_wrong"], name, pol))
    print(f"{name:<22} win {m['win_rate']:.2f}%  strikes {m['mean_wrong']:.3f}"
          f"  [{time.time()-t:.0f}s]")

best_win, _, best_name, best_pol = max(scored, key=lambda r: (r[0], r[1]))
gain = best_win - SOLO_R3
print(f"\nbest: {best_name} -> {best_win:.2f}%   vs round 3 alone {SOLO_R3}%"
      f"   gain {gain:+.2f}")
print("PROCEED" if gain >= 0.3 else "STOP - inside the noise, keep the 70.0334 submission")

## 4. Score on test.txt and write the submission

Runs only if the holdout gain cleared 0.3 points. Submit only if the printed
test.txt number beats **70.097**.

In [ ]:
assert gain >= 0.3, "holdout gain is inside the noise; keep round 3"

metrics = evaluate(test_words, best_pol, max_len=MAX_LEN)
print_report(metrics)
print(f"\nround 3 alone was 70.097% / 3.433 strikes")
print(f"{best_name} is {metrics['win_rate']:.3f}% / {metrics['mean_wrong']:.3f} strikes")

write_submission(metrics["guesses"], "submission.csv")
validate_submission("submission.csv", expected_rows=len(test_words))
json.dump({"best": best_name, "holdout": best_win, "test": metrics["win_rate"]},
          open("/kaggle/working/ensemble_result.json", "w"))